In [1]:
import pandas as pd
import os
import json

In [2]:
results_paths = [
    f"../checkpoints/{dir}/best_history.json" for dir in os.listdir("../checkpoints") if os.path.isdir(os.path.join("../checkpoints", dir))
]
results_paths

['../checkpoints/bigru_finetune/best_history.json',
 '../checkpoints/bigru_fusion_finetune/best_history.json',
 '../checkpoints/bigru_fusion_pretrain/best_history.json',
 '../checkpoints/bigru_pretrain/best_history.json',
 '../checkpoints/cnn_bigru_finetune/best_history.json',
 '../checkpoints/cnn_bigru_pretrain/best_history.json',
 '../checkpoints/cnn_lstm_finetune/best_history.json',
 '../checkpoints/cnn_lstm_fusion_finetune/best_history.json',
 '../checkpoints/cnn_lstm_fusion_pretrain/best_history.json',
 '../checkpoints/cnn_lstm_pretrain/best_history.json',
 '../checkpoints/lstm_finetune/best_history.json',
 '../checkpoints/lstm_pretrain/best_history.json']

In [3]:
def load_experiments(folder):
    rows = []
    for file in results_paths:
        with open(file, "r") as f:
            data = json.load(f)
        experiment = data["experiment"]
        epochs = data["epoch"]
        fields = {
            "train_loss",
            "val_loss",
            "disease_macro_auroc",
            "disease_macro_auprc",
            "disease_macro_f1",
        }
        for i, epoch in enumerate(epochs):
            row = {
                "experiment": experiment,
                "epoch": epoch,
            }
            for field in fields:
                if field in data:
                    row[field] = data[field][i]
            rows.append(row)
    return pd.DataFrame(rows)

In [4]:
loaded_df = load_experiments("../checkpoints")
loaded_df

,experiment,epoch,disease_macro_f1,val_loss,disease_macro_auprc,disease_macro_auroc,train_loss
0,bigru_finetune,1,0.245454,0.774885,0.213370,0.740412,0.959736
1,bigru_finetune,2,0.256710,0.757574,0.223714,0.755577,0.894387
2,bigru_finetune,3,0.255812,0.754203,0.229053,0.761584,0.880100
3,bigru_finetune,4,0.274634,0.734171,0.241768,0.776614,0.860056
4,bigru_finetune,5,0.277782,0.730120,0.243494,0.777230,0.842843
...,...,...,...,...,...,...,...
334,lstm_pretrain,20,0.732030,0.539557,0.773240,0.803467,0.399534
335,lstm_pretrain,21,0.736917,0.545535,0.771683,0.804775,0.390106
336,lstm_pretrain,22,0.728961,0.540739,0.768650,0.801305,0.371917
337,lstm_pretrain,23,0.729106,0.557049,0.766984,0.800220,0.362881


In [5]:
pretrained_df = loaded_df[loaded_df["experiment"].str.contains("pretrain")].reset_index(drop=True)
finetune_df = loaded_df[loaded_df["experiment"].str.contains("finetune")].reset_index(drop=True)

In [6]:
pretrained_df.head()

,experiment,epoch,disease_macro_f1,val_loss,disease_macro_auprc,disease_macro_auroc,train_loss
0,bigru_fusion_pretrain,1,0.730486,0.501708,0.773189,0.807931,0.536380
1,bigru_fusion_pretrain,2,0.732606,0.489908,0.783971,0.817278,0.510121
2,bigru_fusion_pretrain,3,0.739873,0.488853,0.788274,0.819813,0.497975
3,bigru_fusion_pretrain,4,0.745207,0.486536,0.791876,0.821789,0.488914
4,bigru_fusion_pretrain,5,0.743692,0.478853,0.795604,0.824858,0.479627


In [7]:
finetune_df.head()

,experiment,epoch,disease_macro_f1,val_loss,disease_macro_auprc,disease_macro_auroc,train_loss
0,bigru_finetune,1,0.245454,0.774885,0.213370,0.740412,0.959736
1,bigru_finetune,2,0.256710,0.757574,0.223714,0.755577,0.894387
2,bigru_finetune,3,0.255812,0.754203,0.229053,0.761584,0.880100
3,bigru_finetune,4,0.274634,0.734171,0.241768,0.776614,0.860056
4,bigru_finetune,5,0.277782,0.730120,0.243494,0.777230,0.842843


In [8]:
# choose the best epoch based on validation loss for each experiment to decide with which architecture to continue experimenting
pretrained_df = pretrained_df.loc[pretrained_df.groupby("experiment")["val_loss"].idxmin()].reset_index(drop=True)
finetune_df = finetune_df.loc[finetune_df.groupby("experiment")["val_loss"].idxmin()].reset_index(drop=True)

In [9]:
pretrained_df

,experiment,epoch,disease_macro_f1,val_loss,disease_macro_auprc,disease_macro_auroc,train_loss
0,bigru_fusion_pretrain,5,0.743692,0.478853,0.795604,0.824858,0.479627
1,bigru_pretrain,7,0.741419,0.488210,0.789452,0.819542,0.477865
2,cnn_bigru_pretrain,10,0.736554,0.481745,0.797016,0.820939,0.495355
3,cnn_lstm_fusion_pretrain,30,0.748957,0.468683,0.805570,0.831068,0.467828
4,cnn_lstm_pretrain,29,0.746468,0.475344,0.801073,0.827501,0.461839
5,lstm_pretrain,9,0.740170,0.494975,0.785963,0.814603,0.486706


In [10]:
finetune_df

,experiment,epoch,disease_macro_f1,val_loss,disease_macro_auprc,disease_macro_auroc,train_loss
0,bigru_finetune,10,0.285775,0.702793,0.257905,0.789841,0.804815
1,bigru_fusion_finetune,16,0.292041,0.673877,0.290807,0.809152,0.717102
2,cnn_bigru_finetune,23,0.289953,0.687804,0.278759,0.806428,0.790418
3,cnn_lstm_finetune,30,0.290584,0.692087,0.276034,0.799110,0.780090
4,cnn_lstm_fusion_finetune,11,0.297740,0.667641,0.296517,0.812507,0.782701
5,lstm_finetune,8,0.278065,0.719765,0.243290,0.779596,0.824932


In [11]:
# cnn_lstm + bigru

In [12]:
hparams_bigru = pd.read_csv("../results/bigru_pretrain_hparams.csv")
hparams_lstm = pd.read_csv("../results/cnn_lstm_hparams.csv")

In [15]:
hparams_bigru.sort_values(by="target_metric", ascending=True).head()

,trial_number,state,stopped_epoch,target_metric,best_val_loss,best_macro_auroc,pretrain_lr,weight_decay,scheduler_patience,scheduler_factor,tabular_hidden_dim,tabular_fusion_dim,tabular_dropout,hidden_dim,num_layers,rnn_dropout,head_dropout
23,23,COMPLETE,30,0.474380,0.474380,0.0,0.000250,0.000015,5,0.2,64,256,0.4,128,1,0.2,0.3
31,31,COMPLETE,25,0.474819,0.474819,0.0,0.000306,0.000039,4,0.3,256,128,0.4,128,1,0.1,0.4
41,41,PRUNED,15,0.474889,0.474889,0.0,0.000293,0.000022,5,0.2,256,128,0.2,128,2,0.1,0.4
25,25,COMPLETE,15,0.475996,0.475996,0.0,0.000493,0.000025,6,0.3,64,256,0.4,128,1,0.1,0.4
0,0,COMPLETE,23,0.476003,0.476003,0.0,0.000307,0.000037,4,0.4,256,128,0.2,128,1,0.2,0.3


In [16]:
hparams_lstm.sort_values(by="target_metric", ascending=True).head()

,trial_number,state,stopped_epoch,target_metric,best_val_loss,best_macro_auroc,pretrain_lr,weight_decay,scheduler_patience,scheduler_factor,tabular_hidden_dim,tabular_fusion_dim,tabular_dropout,hidden_dim,num_layers,rnn_dropout,head_dropout,cnn_channels,cnn_kernels,cnn_dropout
45,45,COMPLETE,28,0.468323,0.468323,0.0,0.000377,0.000162,3,0.5,256,256,0.3,128,1,0.1,0.3,64_128,15_7,0.2
41,41,COMPLETE,28,0.468568,0.468568,0.0,0.000253,0.000091,3,0.4,256,256,0.1,128,1,0.1,0.3,64_128,15_7,0.2
15,15,COMPLETE,33,0.470364,0.470364,0.0,0.000204,0.000044,3,0.4,256,128,0.1,128,1,0.2,0.2,64_128,15_7,0.3
43,43,COMPLETE,18,0.470371,0.470371,0.0,0.000362,0.000171,3,0.5,256,256,0.1,128,1,0.1,0.3,64_128,15_7,0.1
27,27,COMPLETE,22,0.470559,0.470559,0.0,0.000209,0.000012,8,0.5,64,512,0.3,128,1,0.3,0.4,64_128,15_7,0.3
